<a href="https://colab.research.google.com/github/expely/Business-Analytics-Labs/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

**Important:** Perform this split **before** any preprocessing or feature transformations.

### In Your Response:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

url = 'https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/refs/heads/main/DataSets/hotels.csv'
df = pd.read_csv(url)

print(df.shape)

# Remove missing data columns
df = df.dropna(subset=[
    'country', 'agent', 'company'
])

# Prevent data leakage
df = df.drop(columns=['reservation_status', 'reservation_status_date'])

# Encode Categories
categorical_cols = [
    'hotel', 'arrival_date_month', 'meal', 'country',
    'market_segment', 'distribution_channel', 'reserved_room_type',
    'assigned_room_type', 'deposit_type', 'customer_type', 'agent', 'company'
]
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Split into X and Y
y = df_encoded['is_canceled']
X = df_encoded.drop(columns=['is_canceled'])

# Split into the training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

(9404, 32)


### ✍️ Your Response: 🔧
1. There are 32 columns and 9404 rows in the dataset.

2. I included all of the columns, except for ones with nulls in them (agent and company) as well as any that can cause data leakage (like reservation status, reservation status date, etc.) since they are based on whether or not they are cnacelled. Features are also included for one-hot encoding.

3. I first looked at all the data with df.info, them I dropped the missing data columns, removed the leaked data, encoded categories, and split them into the training and testing data set.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Make sure to split your data first (see the previous step), then fit any text/vector preprocessing on training data only.
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

**Note:** If you use a vectorizer (e.g., `CountVectorizer`), fit it on the training data only, then transform both training and test data.

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Fit model
nb_model = BernoulliNB()
nb_model.fit(X_train, y_train)

# Predict and evaluate
y_pred_nb = nb_model.predict(X_test)

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_nb))
nb_acc = accuracy_score(y_test, y_pred_nb)
print("Accuracy:", nb_acc)

Confusion Matrix:
 [[31  4]
 [ 6  0]]

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.89      0.86        35
           1       0.00      0.00      0.00         6

    accuracy                           0.76        41
   macro avg       0.42      0.44      0.43        41
weighted avg       0.72      0.76      0.74        41

Accuracy: 0.7560975609756098


### ✍️ Your Response: 🔧
1. Overall it performed well, with a 75% accuracy (true positive and true negative %). One of the best ones is going to be recall, we want to know out of all those who actually cancelled, what did our model catch? It looks like we're really good at predicting when they are not going to cancel, with an 89% in recall.

2. This can be used to plan how much, and how we overbook during certain seasons. If we know about 89% are not going to cancel in a certain period, we can then overbook by 11% to hopefully stay at full capacity.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Scale the data using `StandardScaler` to bring large numbers into a smaller range (Note: use a scaled training set, but fit the scaler only on X_train).
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?


In [22]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# Add numeric feature
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_encoded.drop(columns=['is_canceled']))

# Split
X_train_svm, X_test_svm, y_train_svm, y_test_svm = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

# Train SVM
svm_model = SVC(kernel='linear', C=1.0, gamma='scale')
svm_model.fit(X_train_svm, y_train_svm)

# Predict
y_pred_svm = svm_model.predict(X_test_svm)

# Evaluate
print("Confusion Matrix:\n", confusion_matrix(y_test_svm, y_pred_svm))
print("\nClassification Report:\n", classification_report(y_test_svm, y_pred_svm))
svm_acc = accuracy_score(y_test_svm, y_pred_svm)
print("Accuracy:", svm_acc)

Confusion Matrix:
 [[32  3]
 [ 4  2]]

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.91      0.90        35
           1       0.40      0.33      0.36         6

    accuracy                           0.83        41
   macro avg       0.64      0.62      0.63        41
weighted avg       0.82      0.83      0.82        41

Accuracy: 0.8292682926829268


### ✍️ Your Response: 🔧
1. The model performs better than the previous with an accuracy of 83%. This one has a lot better recall for accurately predicting when a guest will actually cancel, at a 33% (compared to the Naive Bayes being a 0%). False positives are costly here, so that's going to be the best judge.

2. We want to use it when there are complex relationships between the data. A good example would be the adr and type of guest. A corporate guest likely doesn't care about the adr, but a regular guest is going to be a lot more sensitive to that kind of thing. The SVM models are a lot better at catching those cases.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Use a true validation split from the training data, not the test set, for validation_data
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

### In Your Response:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?


In [23]:
from sklearn.neural_network import MLPClassifier

# Model
model = MLPClassifier(hidden_layer_sizes=(100,), early_stopping=True, random_state=42)

# Fit the Model
history = model.fit(X_scaled, y)

# Predict and evaluate
y_pred_nn = (model.predict(X_test_svm) > 0.5).astype(int)
print("Confusion Matrix:\n", confusion_matrix(y_test_svm, y_pred_nn))
print("\nClassification Report:\n", classification_report(y_test_svm, y_pred_nn))
neural_acc = accuracy_score(y_test_svm, y_pred_nn)
print("Accuracy:", neural_acc)

Confusion Matrix:
 [[35  0]
 [ 3  3]]

Classification Report:
               precision    recall  f1-score   support

           0       0.92      1.00      0.96        35
           1       1.00      0.50      0.67         6

    accuracy                           0.93        41
   macro avg       0.96      0.75      0.81        41
weighted avg       0.93      0.93      0.92        41

Accuracy: 0.926829268292683


### ✍️ Your Response: 🔧
1. It is by far the best one with an accuracy of 93%, and high recall scores.

2. I think for hotels, they would likely be fine with it, as this is mostly for internal use of predicting whether someone will classify it or not. Marketing could even use a "neural network AI" buzz word in some of their marketing prompotions. However, if they had to report to some government stakeholder for compliance and regulations, they tend to be pretty unhappy with hand-wavy black box scenarios.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

### In Your Response:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?


In [24]:
print("Naive Bayes Accuracy:", nb_acc)
print("\nSVM Accuracy:", svm_acc)
print("\nNeural Network Accuracy:", neural_acc)

Naive Bayes Accuracy: 0.7560975609756098

SVM Accuracy: 0.8292682926829268

Neural Network Accuracy: 0.926829268292683


### ✍️ Your Response: 🔧
1. Best Overall Accuracy: Neural Network with a 93%. Training Time: Naive Bayes. Interpretability: SVM, because higher ups tend to like a little stroke of the ego that this model captures complex relationships we humans naturally understand. Ease of Use: Neural Network, I would usually say Naive Bayes, but I'd argue in order to get a NB model working, it would take just as much effort to get the MPClassifier to work. And, the MPClassifier just works really well.

2. I would, since it's an internal metric we can use for strategy, it's not pivotal for people to understand the black box-iness of the neural network. It captures the predictions a lot better. It would take a lot longer to train, but not too much longer for it to matter for our business use case.

## 6. Final Business Recommendation

### In Your Response:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?
2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. I would recommend to use the Neural Network model. It helps the business to understand when a customer is going to cancel their booking, or not. The biggest risk with this model is the length of time to train, as we get more data to train the model, it takes longer to actually train. But, we're currently talking a matter of minutes. To make this data more accurate, it would be nice to have a lot more data for us to use, rather than ~10k rows only.

2. This is incredible, it's the whole shebang of going from nothing, to cleaning, exploring the dataset a little bit, removing variables that have leakage, etc. And I can go back and see exactly what I did and present it to leaders in a notebook! It's really really awesome to have learned all this business analytics with Python.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [25]:
!jupyter nbconvert --to html "assignment_12_bayes_svm_neural.ipynb"

[NbConvertApp] Converting notebook assignment_12_bayes_svm_neural.ipynb to html
[NbConvertApp] Writing 321030 bytes to assignment_12_bayes_svm_neural.html
